In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "retail_sales"

bronze_df = spark.table(
    f"{CATALOG}.{SCHEMA}.bronze_sales"
)

print("Total rows:", bronze_df.count())
display(bronze_df.limit(10))

In [0]:
bronze_df.printSchema()

print("Total columns:", len(bronze_df.columns))

print("Column names:")
for column in bronze_df.columns:
  print(column)

In [0]:
sales_selected_df = bronze_df.select(
    "ss_sold_date_sk",
    "ss_item_sk",
    "ss_customer_sk",
    "ss_ticket_number",
    "ss_quantity",
    "ss_sales_price",
    "ss_net_paid"
)

display(sales_selected_df.limit(10))

In [0]:
high_value_df = sales_selected_df.filter(
    F.col("ss_net_paid") > 100
)

display(high_value_df.limit(10))

print("High-value rows:", high_value_df.count())

In [0]:
sales_transformed_df = sales_selected_df.withColumn(
    "gross_sales_at_selling_price",
    F.col("ss_quantity") * F.col("ss_sales_price")
)

display(
    sales_transformed_df.select(
        "ss_quantity",
        "ss_sales_price",
        "gross_sales_at_selling_price",
        "ss_net_paid"
    ).limit(10)
)

In [0]:
highest_sales_df = sales_transformed_df.orderBy(
    F.col("ss_net_paid").desc()
)

display(highest_sales_df.limit(10))

In [0]:
store_sales_summary_df = (
    sales_transformed_df
    .groupBy("ss_customer_sk")
    .agg(
        F.count("*").alias("sales_line_count"),
        F.sum("ss_quantity").alias("total_quantity"),
        F.round(F.sum("ss_net_paid"),2).alias("total_revenue"),
        F.round(F.avg("ss_net_paid"), 2).alias("avg_line_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

display(store_sales_summary_df)

In [0]:
#Read product information
items_df = spark.table(
    "samples.tpcds_sf1.item"
)

items_selected_df = items_df.select(
    "i_item_sk",
    "i_item_id",
    "i_product_name",
    "i_category",
    "i_class"
)

display(items_selected_df.limit(10))

In [0]:
sales_products_df = (
    sales_transformed_df.alias("s")
    .join(
        items_selected_df.alias("i"),
        F.col("s.ss_item_sk") == F.col("i.i_item_sk"),
        "inner"
    )
    .select(
        F.col("s.ss_sold_date_sk"),
        F.col("s.ss_ticket_number"),
        F.col("s.ss_customer_sk"),
        F.col("i.i_product_name").alias("product_name"),
        F.col("i.i_category").alias("product_category"),
        F.col("s.ss_quantity").alias("quantity"),
        F.col("s.ss_net_paid").alias("net_revenue")
    )
)

display(sales_products_df.limit(10))

In [0]:
category_summary_df = (
    sales_products_df
    .groupBy("product_category")
    .agg(
        F.round(
            F.sum("net_revenue"),2
        ).alias("total_revenue"),

        F.sum("quantity").alias("total_quantity"),
        F.count("*").alias("sales_line_count")
    )
    .orderBy(
        F.col("total_revenue").desc()
    )
)

display(category_summary_df)

In [0]:
columns_to_check = [
    "ss_item_sk",
    "ss_customer_sk",
    "ss_quantity",
    "ss_net_paid"
]

null_expressions = [
    F.sum(
        F.col(column_name).isNull().cast("int")
    ).alias(column_name)
    for column_name in columns_to_check
]

display(
    bronze_df.select(null_expressions)
)

In [0]:
unmatched_sales_df = (
    sales_transformed_df.alias("s")
    .join(
        items_selected_df.alias("i"),
        F.col("s.ss_item_sk") == F.col("i.i_item_sk"),
        "left_anti"
    )
)

print(
    "Sales rows without matching products:",
    unmatched_sales_df.count()
)

In [0]:
category_summary_df.createOrReplaceTempView(
    "vw_category_summary"
)

print("Temporary SQL view created.")

In [0]:
%sql
SELECT
    product_category,
    total_revenue,
    total_quantity,
    sales_line_count
FROM vw_category_summary
ORDER BY total_revenue DESC;